# Figure S3b,c. Sketches for a smaller study sample

Design from the MSK notebook `/home/sunh4/rss_qc/2026_Simulation/1.Stoc_LD_validation.ipynb` ("2. Oversaturating model"), split into data and fine-mapping steps as in `04` / `05`. Compute summary statistics from 2,000 UK Biobank participants and draw the LD sketch and random subsets from the full pool of 342,053. The gains from increasing the sketch dimension level off sooner for the smaller study.

- `[target_ld]`: per region, 2,000 target individuals (fixed seed, shared by every h2g / causal setting) and their in-sample LD.
- `[target_pheno]`: `04` `[gtex_pheno]` on the target individuals; h2g 0.06 / 0.09 / 0.12 → NCP = n × h2g = 120 / 180 / 240 (Fig 2F labels); 1–3 causal.
- Sketch and subset LD: `04` `[gtex_ld]` output (`UKBB_sketch_dimension/result/B_<B>/ld`, B = 500–20,000), not recomputed; they do not depend on the target.
- `[ld_region_map]`: region id → `04` LD file name (`04` names files by position in its `--genofile` list); files here are named by region id (`UKBB_<i>`).
- `[target_finemap]`: `05` `[dim_finemap]` (same 5 fits) with the target LD as in-sample LD.
- `[target_LD_mismatch]`: `04` `[LD_mismatch]` (MSE, median sq. error, max |ΔLD|, concordance, D2) with the target LD as the truth.
- Summary: `05` `[susie_cs_summary]`.

Outputs (`/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/`):

- `UKBB_B_Over_Saturate/result/` (target, pheno)
- `UKBB_B_Over_Saturate/ld_mismatch/ld_mismatch_combined.rds` (Figure S3b)
- `UKBB_B_Over_Saturate_finemap/summary/result_susie_cs_summary_summary.rds` (Figure S3c)

## 1. Simulate data

In [ ]:
[target_ld]
# Target (GWAS) individuals: sample_size random UKB individuals per region (fixed seed), and their in-sample LD.
parameter: genofile = paths
parameter: cwd = path("output")
parameter: job_size = 10
parameter: walltime = "100h"
parameter: mem = "60G"
parameter: numThreads = 2
parameter: sample_size = 2000
parameter: seed = 2026
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/target/{_input:bn}_target_n_{sample_size}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)

    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)

    ## 1. Load genotype (unrelated EUR, trimmed) ------------------------------
    file <- ${_input:r}
    i <- file %>% str_remove("\\.bim$") %>% str_remove(".*/UKBB_")
    UKBB_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed/result/UKBB_", i)
    UKBB_dat <- plink_process(UKBB_path)
    Xmat <- UKBB_dat$bed
    if (ncol(Xmat) < 500) stop("Too few SNPs — terminating job")

    ## 2. Target individuals (same for every h2g / n_causal) and their LD ------
    set.seed(${seed} + as.integer(i))           # per region, independent of the --genofile order
    idx_target <- sort(sample(nrow(Xmat), ${sample_size}, replace = FALSE))
    X_target   <- Xmat[idx_target, , drop = FALSE]
    LD_full    <- get_correlation(X_target)   # in-sample LD of the GWAS individuals

    ## 3. Save ----------------------------------------------------------------
    target_data <- list(
      region_id  = i,
      idx_target = idx_target,
      LD_full    = LD_full,
      n          = nrow(X_target),
      p          = ncol(X_target),
      n_ref      = nrow(Xmat),                # individuals behind the sketch / subset LD (04)
      mat_dim    = dim(X_target)
    )
    dir.create(dirname(${_output:r}), recursive = TRUE, showWarnings = FALSE)
    saveRDS(target_data, ${_output:r})

In [ ]:
[target_pheno]
# 04 [gtex_pheno] on the target individuals from [target_ld]. Loads genotype but stores NO LD.
parameter: genofile = paths
parameter: target_dir = path
# [target_ld] output folder (…/result/target)
parameter: sample_size = 2000
parameter: cwd = path("output")
parameter: job_size = 10
parameter: walltime = "100h"
parameter: mem = "120G"
parameter: numThreads = 2
parameter: n_causal = 1
parameter: h2g = 0.05
parameter: total_h2g = True
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/pheno/{_input:bn}_ncausal_{n_causal}_h2g_{h2g}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)

    ## 1. Load genotype (unrelated EUR, trimmed) ------------------------------
    file <- ${_input:r}
    i <- file %>% str_remove("\\.bim$") %>% str_remove(".*/UKBB_")
    UKBB_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed/result/UKBB_", i)
    UKBB_dat <- plink_process(UKBB_path)
    if (ncol(UKBB_dat$bed) < 500) stop("Too few SNPs — terminating job")

    # target individuals from [target_ld], matched by region id
    target <- readRDS(file.path(${target_dir:r}, paste0("UKBB_", i, "_target_n_", ${sample_size}, ".rds")))
    stopifnot(target$region_id == i)
    Xmat <- UKBB_dat$bed[target$idx_target, , drop = FALSE]

    n_samples <- nrow(Xmat); p <- ncol(Xmat)

    ## 2. Simulate phenotype (target individuals only) ------------------------
    ntrait  <- 1
    ncausal <- ${n_causal}
    h2g     <- ${h2g}

    causal_index <- sample_causal_index(Xmat = Xmat, ncausal = ncausal)
    B_mat <- matrix(0, nrow = p, ncol = ntrait)
    beta  <- sim_beta_fix_variant(G = Xmat, causal_index = causal_index, is_h2g_total = FALSE)
    B_mat[, 1] <- beta

    phenotype <- sim_multi_traits(G = Xmat, B = B_mat, h2g = h2g, is_h2g_total = FALSE)$P

    ## 3. Summary stats ------------------------------------------------------
    target_tb  <- calculate_sumstat(Xmat, as.numeric(phenotype[, 1]))
    sumstat_qc <- target_tb
    z_vec      <- sumstat_qc$z

    ## 4. Save (phenotype + sumstats only; index-matches the LD rds) ----------
    pheno_data <- list(
      region_id  = i,
      sumstat_qc = sumstat_qc,
      z_vec      = z_vec,
      causal_index = causal_index,
      true_vars  = causal_index,
      beta       = beta,
      phenotype  = phenotype,
      n          = n_samples,                # = sample_size
      p          = p,
      h2g        = h2g,
      n_causal   = ncausal
    )
    dir.create(dirname(${_output:r}), recursive = TRUE, showWarnings = FALSE)
    saveRDS(pheno_data, ${_output:r})

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="UKBB_B_Over_Saturate"
job_dir="${base_dir}/${job_name}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

work_dir="${job_dir}"

# ============================================================
# PART 1 — target individuals + their LD (once per region)
# ============================================================
cat << 'EOF' > ${job_name}_target.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME_target
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=60000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME_target.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME_target.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/26.FigS3B_S3C.Over_Saturated_Sketch.ipynb" target_ld \
    --genofile $(echo /sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed/result/*.bim) \
    --sample_size 2000 \
    --cwd WORK_DIR/result/ -j 5
EOF

sed -e "s|WORK_DIR|${work_dir}|g" -e "s|JOB_NAME|${job_name}|g" ${job_name}_target.sh > ${job_name}_target.sh.submit
sb ${job_name}_target.sh.submit


Submit after the `[target_ld]` jobs finish (the phenotypes are simulated on the saved target individuals).

In [ ]:
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="UKBB_B_Over_Saturate"
job_dir="${base_dir}/${job_name}"
cd "${job_dir}/code"
work_dir="${job_dir}"

# ============================================================
# PART 2 — phenotype + sumstats (loop over causal x h2g; after PART 1)
# ============================================================
cat << 'EOF' > ${job_name}_pheno.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME_ph_c_CAUSAL_h_H2G
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=60000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME_ph_c_CAUSAL_h_H2G.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME_ph_c_CAUSAL_h_H2G.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/26.FigS3B_S3C.Over_Saturated_Sketch.ipynb" target_pheno \
    --genofile $(echo /sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed/result/*.bim) \
    --target_dir WORK_DIR/result/target --sample_size 2000 \
    --n_causal CAUSAL --h2g H2G \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL/ -j 5
EOF

chmod +x ${job_name}_pheno.sh

# n = 2000; NCP = n x h2g = 120 / 180 / 240 (the NCP labels of Fig 2F)
for causal in 1 2 3; do
  for h2g in 0.06 0.09 0.12; do
      output_script="${job_name}_pheno_causal_${causal}_h2g_${h2g}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${job_name}_pheno.sh" > "${output_script}"

       sb "${output_script}"
  done
done

## 2. Fine-mapping

Map region id → `04` LD file once (a few minutes; run interactively or as a short job) before the fine-mapping jobs.

In [ ]:
[ld_region_map]
# 04 [gtex_ld] names its LD files region_<_index> by position in its --genofile list. 04's pheno files use the
# same index and store region_id, so read them (small) once to map region id -> 04 file name, then check a few
# 04 LD files against the map.
parameter: pheno_dir = path
# one 04 pheno folder, e.g. UKBB_sketch_dimension/result/h2g_0.0006/causal_1/pheno
parameter: ld_dir = path
# one 04 LD folder, e.g. UKBB_sketch_dimension/result/B_500/ld
parameter: n_check = 5
parameter: cwd = path("output")
parameter: container = ""
output: f'{cwd:a}/ld_region_map.tsv'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    files <- list.files(${pheno_dir:r}, pattern = "^region_[0-9]+_.*\\.rds$", full.names = TRUE)
    if (!length(files)) stop("no 04 pheno files in ", ${pheno_dir:r})
    map <- do.call(rbind, lapply(files, function(f) data.frame(
      region_tok = regmatches(basename(f), regexpr("region_[0-9]+", basename(f))),
      region_id  = as.character(readRDS(f)$region_id))))
    stopifnot(nrow(map) > 0, !anyDuplicated(map$region_tok), !anyDuplicated(map$region_id))

    ## the LD files are large: check a few spread across the list
    ld_files <- list.files(${ld_dir:r}, pattern = "^region_[0-9]+_.*\\.rds$", full.names = TRUE)
    for (f in ld_files[unique(round(seq(1, length(ld_files), length.out = min(${n_check}, length(ld_files)))))]) {
      tok <- regmatches(basename(f), regexpr("region_[0-9]+", basename(f)))
      stopifnot(readRDS(f)$region_id == map$region_id[map$region_tok == tok])
      cat("checked", basename(f), "\n")
    }
    write.table(map, ${_output:r}, sep = "\t", quote = FALSE, row.names = FALSE)
    cat(nrow(map), "regions mapped\n")

In [ ]:
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/26.FigS3B_S3C.Over_Saturated_Sketch.ipynb" ld_region_map \
    --pheno_dir /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_sketch_dimension/result/h2g_0.0006/causal_1/pheno \
    --ld_dir /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_sketch_dimension/result/B_500/ld \
    --cwd /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B_Over_Saturate

In [ ]:
[target_finemap]
# 05 [dim_finemap]: in-sample LD = target LD from [target_ld]; sketch / subset LD = 04 [gtex_ld] (all 342,053).
parameter: pheno_files = paths         
parameter: ld_dir = path               
# 04 output: UKBB_sketch_dimension/result/B_<B>/ld
parameter: target_dir = path
# [target_ld] output folder (…/result/target)
parameter: ld_map = path
# [ld_region_map] output: region id -> 04 LD file name
parameter: B_stoc = 10000               
parameter: cwd = path("output")
parameter: job_size = 10
parameter: walltime = "72h"
parameter: mem = "60G"
parameter: numThreads = 2
parameter: container = ""
input: pheno_files, group_by = 1
output: f'{cwd:a}/finemap/{_input:bn}_B_stoc_{B_stoc}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(Rfast)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
    
    # Project utilities
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")
    
    # -----------------------------------------------------------------------------
    # Two susieR builds, each in its own environment so they never clobber.
    # Each susie_*() carries its enclosing env, so internal helper lookups resolve
    # against the build it came from.
    #   susie_hc : Haochen multi-panel / R_finite build -> vanilla + _pop (eb)
    #   susie_xw : Xuewei ld_mismatch build             -> _mix  (eb_mix)
    # -----------------------------------------------------------------------------
    source_build <- function(dir) {
      env <- new.env(parent = globalenv())
      for (f in list.files(dir, pattern = "\\.R$", full.names = TRUE)) sys.source(f, envir = env)
      env
    }
    susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss

    ## ---- safe wrapper (yours) ----------------------------------------------
    safe_susie <- function(.fn, .label, ...) {
      tryCatch(
        withCallingHandlers(.fn(...), warning = function(w) invokeRestart("muffleWarning")),
        error = function(e) { cat(sprintf("[FAIL] %s: %s\n", .label, conditionMessage(e))); NULL }
      )
    }

    ## ---- load matched pheno + LD -------------------------------------------
    pheno_path <- ${_input:r}
    pheno_dat  <- readRDS(pheno_path)

    # 04 names its LD files by position (region_<_index>); find this region's file by region id
    ld_map     <- read.delim(${ld_map:r}, colClasses = "character")
    region_tok <- ld_map$region_tok[ld_map$region_id == as.character(pheno_dat$region_id)]
    stopifnot(length(region_tok) == 1)
    ld_path    <- file.path("${ld_dir}", paste0(region_tok, "_ukb_B_stoc_", ${B_stoc}, ".rds"))
    stopifnot(file.exists(ld_path))
    ld_dat     <- readRDS(ld_path)

    target_path <- file.path("${target_dir}", paste0("UKBB_", pheno_dat$region_id, "_target_n_", pheno_dat$n, ".rds"))
    stopifnot(file.exists(target_path))
    target      <- readRDS(target_path)
    stopifnot(ld_dat$region_id == pheno_dat$region_id, target$region_id == pheno_dat$region_id)

    z        <- pheno_dat$z_vec               # == sumstat_qc$z
    n        <- pheno_dat$n                    # GWAS sample size (target individuals)
    true_vars<- pheno_dat$true_vars

    LD_full  <- target$LD_full                 # in-sample LD of the target individuals
    LD_sub   <- ld_dat$LD_sample
    U        <- ld_dat$U                       # p x B_stoc
    B_sketch <- ncol(U)                        # = ld_dat$B_stoc, sketch dimension
    n_sub    <- length(ld_dat$idx_ld)          # subset sample size behind LD_sample

    ## ---- 5 finemapping runs -------------------------------------------------
    # 1. in-sample LD (exact)
    fit_insample   <- safe_susie(susie_hc, "insample",   z = z, R = LD_full, n = n)

    # 2. subset LD (naive, no correction)
    fit_subset     <- safe_susie(susie_hc, "subset",     z = z, R = LD_sub,  n = n)

    # 3. sketch (naive, U as input, no correction)
    fit_sketch     <- safe_susie(susie_hc, "sketch",     z = z, X = t(U),    n = n)

    # 4. sketch + correction   -> R_finite = sketch dimension
    fit_sketch_mix <- safe_susie(susie_hc, "sketch_mix", z = z, X = t(U),    n = n,
                                 R_finite = B_sketch)

    # 5. subset LD + correction -> R_finite = subset sample size
    fit_subset_mix <- safe_susie(susie_hc, "subset_mix", z = z, R = LD_sub,  n = n,
                                 R_finite = n_sub)

    ## ---- save ---------------------------------------------------------------
    finemap_out <- list(
      region_id      = pheno_dat$region_id,
      h2g            = pheno_dat$h2g,
      n_causal       = pheno_dat$n_causal,
      B_stoc         = B_sketch,
      n_sub          = n_sub,
      true_vars      = true_vars,
      fit_insample   = fit_insample,
      fit_subset     = fit_subset,
      fit_sketch     = fit_sketch,
      fit_sketch_mix = fit_sketch_mix,
      fit_subset_mix = fit_subset_mix
    )
    dir.create(dirname(${_output:r}), recursive = TRUE, showWarnings = FALSE)
    saveRDS(finemap_out, ${_output:r})

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
data_dir="${base_dir}/UKBB_B_Over_Saturate"
job_name="UKBB_B_Over_Saturate_finemap"
job_dir="${base_dir}/${job_name}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

work_dir="${job_dir}"

# ===== BSUB template for the finemap step =====
cat << 'EOF' > ${job_name}_finemap.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME_fm_c_CAUSAL_h_H2G_B_B_STOC
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=60000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME_fm_c_CAUSAL_h_H2G_B_B_STOC.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME_fm_c_CAUSAL_h_H2G_B_B_STOC.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/26.FigS3B_S3C.Over_Saturated_Sketch.ipynb" target_finemap \
    --pheno_files $(echo DATA_DIR/result/h2g_H2G/causal_CAUSAL/pheno/*.rds) \
    --ld_dir /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_sketch_dimension/result/B_B_STOC/ld \
    --target_dir DATA_DIR/result/target \
    --ld_map DATA_DIR/ld_region_map.tsv \
    --B_stoc B_STOC \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL/B_B_STOC/ -j 5
EOF

chmod +x ${job_name}_finemap.sh

for causal in 1 2 3; do
  for h2g in 0.06 0.09 0.12; do
    for B_stoc in 500 1000 2000 3000 5000 10000 20000; do
        output_script="${job_name}_finemap_causal_${causal}_h2g_${h2g}_B_stoc_${B_stoc}.sh"
        sed -e "s|CAUSAL|${causal}|g" \
            -e "s|H2G|${h2g}|g" \
            -e "s|B_STOC|${B_stoc}|g" \
            -e "s|WORK_DIR|${work_dir}|g" \
            -e "s|DATA_DIR|${data_dir}|g" \
            -e "s|JOB_NAME|${job_name}|g" \
          "${job_name}_finemap.sh" > "${output_script}"

         sb "${output_script}"
    done
  done
done

Summary (`05` `[susie_cs_summary]`)

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B_Over_Saturate_finemap/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J over_saturate_summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/05.Fig2B.Sketch_Dimension_Finemap.ipynb" susie_cs_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary


EOF

base_sh="base_script"

output_script="summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}

## 3. LD mismatch vs. the target LD

In [ ]:
[target_LD_mismatch]
# 04 [LD_mismatch]: 04's sketch / subset LD compared with the target LD from [target_ld].
parameter: resfile = paths
# All 04 LD .rds files across B settings; combined into ONE output table.
parameter: target_dir = path
# [target_ld] output folder (…/result/target)
parameter: sample_size = 2000
parameter: cwd = path("output")
parameter: job_size = 10
parameter: walltime = "100h"
parameter: mem = "72G"
parameter: numThreads = 3
parameter: container = ""
# ---- Metric knobs ------------------------------------------------------------
parameter: ld_thresh = 0.05    # keep off-diagonal pairs with |R_insample| > this
parameter: min_pairs = 10      # skip references with fewer kept pairs
input: resfile
output: f'{cwd:a}/ld_mismatch_combined.rds', f'{cwd:a}/ld_mismatch_combined.csv'
task: trunk_workers = 1, walltime = walltime, mem = mem, cores = numThreads,
      tags = f'{step_name}_{_output[0]:bn}'
R:  expand = '${ }', stdout = f"{_output[0]:n}.stdout", stderr = f"{_output[0]:n}.stderr", container = container

    suppressMessages(library(parallel))
    `%||%` <- function(a, b) if (is.null(a)) b else a
    files <- c(${_input:r,})

    cov2cor_safe <- function(M) {
      d <- diag(M); if (any(!is.finite(d)) || any(d <= 0)) return(M); cov2cor(M)
    }

    ## per-region worker: 2-row df (sketch/subset) vs target in-sample LD, or NULL on error
    process_one <- function(f) {
      tryCatch({
        res     <- readRDS(f)
        target  <- readRDS(file.path(${target_dir:r}, paste0("UKBB_", res$region_id, "_target_n_", ${sample_size}, ".rds")))
        stopifnot(target$region_id == res$region_id)
        R_ins   <- cov2cor_safe(target$LD_full)
        p       <- nrow(R_ins)

        # sketch: U is p x B  ->  R_hat = cov2cor(U U^T / B)
        U <- res$U; if (nrow(U) != p) U <- t(U)
        R_sketch <- cov2cor(tcrossprod(U) / ncol(U))
        # subset: dense LD from n_sub individuals
        R_sub    <- cov2cor_safe(res$LD_sample)

        metric <- function(R_ref, B) {
          ut  <- upper.tri(R_ins)
          ins <- R_ins[ut]; ref <- R_ref[ut]
          ok  <- is.finite(ins) & is.finite(ref) & (abs(ins) > ${ld_thresh})
          if (sum(ok) < ${min_pairs})
            return(c(mse = NA_real_, med_sq = NA_real_, max_abs_diff = NA_real_,
                     concord = NA_real_, D2 = NA_real_, n_pairs = sum(ok)))
          d <- ref[ok] - ins[ok]
          c(mse          = mean(d^2),
            med_sq       = median(d^2),
            max_abs_diff = max(abs(d)),
            concord      = suppressWarnings(cor(ref[ok], ins[ok])),
            D2           = if (is.finite(B)) mean(d^2) - mean((1 + ref[ok]^2) / B) else NA_real_,
            n_pairs      = sum(ok))
        }

        B_sketch <- ncol(U)
        H <- list(sketch = list(R = R_sketch, B = B_sketch),
                  subset = list(R = R_sub,    B = NA_real_))   # set B = length(res$idx_ld) to enable subset D2

        m <- t(sapply(H, function(h) metric(h$R, h$B)))
        data.frame(file = basename(f), reference = names(H), m,
                   p = p, B = sapply(H, `[[`, "B"),
                   B_setting = res$B_stoc %||% B_sketch, n_target = target$n,
                   row.names = NULL, check.names = FALSE)
      }, error = function(e) { message("FAILED: ", basename(f), " : ", conditionMessage(e)); NULL })
    }

    parts <- mclapply(files, process_one, mc.cores = ${numThreads})
    ok    <- !vapply(parts, is.null, logical(1))
    dat   <- do.call(rbind, parts[ok])

    saveRDS(dat, ${_output[0]:r})
    write.csv(dat, ${_output[1]:r}, row.names = FALSE)
    cat(sprintf("combined %d/%d regions -> %d rows\n", sum(ok), length(files), nrow(dat)))

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="UKBB_B_Over_Saturate_LD_mismatch"
job_dir="${base_dir}/${job_name}"

mkdir -p "${job_dir}"/{code,log}
cd "${job_dir}/code"
work_dir="${job_dir}"

cat << EOF > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J ${job_name}
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=24000]
#BSUB -W 100:00
#BSUB -o ${work_dir}/log/${job_name}.%J.out
#BSUB -e ${work_dir}/log/${job_name}.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:\$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/26.FigS3B_S3C.Over_Saturated_Sketch.ipynb" target_LD_mismatch \\
    --resfile \$(echo /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_sketch_dimension/result/B_*/ld/*.rds) \\
    --target_dir /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B_Over_Saturate/result/target \\
    --sample_size 2000 \\
    --cwd     /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B_Over_Saturate/ld_mismatch
EOF

chmod +x ${job_name}.sh
sb ${job_name}.sh